# Mean Aspect-F1 Bootstrap Confidence Intervals

This notebook computes the Table 4 mean weighted-F1 and macro-F1 confidence
intervals for ALBERT and BioBERT. Each bootstrap replicate resamples held-out
posts once and applies the same sampled rows to all six aspect heads.

Run all cells in Colab. The final output is a four-row aggregate table; no
row-level predictions are displayed.


In [1]:
# Colab setup and Drive paths
%pip -q install pandas numpy scipy scikit-learn matplotlib openpyxl

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json

import pandas as pd
from IPython.display import Image, display

PROJECTS_ROOT = Path("/content/drive/MyDrive/NLP_Projects")
MENTAL_HEALTH_ROOT = PROJECTS_ROOT / "02_MentalHealth"
ANALYSIS_ROOT = MENTAL_HEALTH_ROOT / "03_R2"
ASPECT_PREDICTIONS = ANALYSIS_ROOT / "06_AspectLabel/outputs/test_predictions_aspect.csv"
ASPECT_CI_OUTPUT = ANALYSIS_ROOT / "07_ValidationRobustness/03_aspect_mean_f1_ci/outputs/aspect_mean_f1_bootstrap.csv"
ASPECT_CI_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
if not ASPECT_PREDICTIONS.exists():
    raise FileNotFoundError(ASPECT_PREDICTIONS)
print("Held-out aspect predictions: found")


Mounted at /content/drive
Held-out aspect predictions: found


In [2]:
"""Compute synchronized row-bootstrap CIs for Table 4 mean F1 values."""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


# Paths are configured in the notebook setup cell.
PREDICTIONS = ASPECT_PREDICTIONS
OUTPUT = ASPECT_CI_OUTPUT
ASPECTS = [
    "depression",
    "anxiety",
    "suicidal",
    "stress",
    "bipolar",
    "personality_disorder",
]
MODELS = ["albert", "biobert"]
AVERAGES = ["weighted", "macro"]
N_RESAMPLES = 1_000
SEED = 2025


def mean_f1(frame: pd.DataFrame, model: str, average: str, indices=None) -> float:
    scores = []
    for aspect in ASPECTS:
        truth = frame[f"{aspect}_true"].to_numpy()
        pred = frame[f"{model}_{aspect}_pred"].to_numpy()
        if indices is not None:
            truth = truth[indices]
            pred = pred[indices]
        scores.append(f1_score(truth, pred, average=average))
    return float(np.mean(scores))


def main() -> None:
    frame = pd.read_csv(PREDICTIONS)
    rng = np.random.RandomState(SEED)
    samples = [rng.randint(0, len(frame), len(frame)) for _ in range(N_RESAMPLES)]
    rows = []
    for model in MODELS:
        for average in AVERAGES:
            observed = mean_f1(frame, model, average)
            replicates = np.asarray(
                [mean_f1(frame, model, average, indices) for indices in samples]
            )
            low, high = np.percentile(replicates, [2.5, 97.5])
            rows.append(
                {
                    "model": model,
                    "metric": f"mean_{average}_f1_across_six_heads",
                    "estimate": observed,
                    "ci_low": low,
                    "ci_high": high,
                    "confidence_level": 0.95,
                    "bootstrap_resamples": N_RESAMPLES,
                    "bootstrap_seed": SEED,
                    "bootstrap_unit": "held-out test post shared across six heads",
                    "n_test_posts": len(frame),
                }
            )
    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(OUTPUT, index=False)
    print(f"Wrote {OUTPUT}")


In [3]:
main()
print("Shared-post bootstrap completed.")


Wrote /content/drive/MyDrive/NLP_Projects/02_MentalHealth/03_R2/07_ValidationRobustness/03_aspect_mean_f1_ci/outputs/aspect_mean_f1_bootstrap.csv
Shared-post bootstrap completed.


In [4]:
result = pd.read_csv(ASPECT_CI_OUTPUT)
display(result.round(6))


,model,metric,estimate,ci_low,ci_high,confidence_level,bootstrap_resamples,bootstrap_seed,bootstrap_unit,n_test_posts
0,albert,mean_weighted_f1_across_six_heads,0.862749,0.859851,0.865292,0.95,1000,2025,held-out test post shared across six heads,10550
1,albert,mean_macro_f1_across_six_heads,0.593096,0.585867,0.600990,0.95,1000,2025,held-out test post shared across six heads,10550
2,biobert,mean_weighted_f1_across_six_heads,0.857260,0.854305,0.859931,0.95,1000,2025,held-out test post shared across six heads,10550
3,biobert,mean_macro_f1_across_six_heads,0.589096,0.581961,0.596771,0.95,1000,2025,held-out test post shared across six heads,10550
